# 7T quantitative figure

Panels a and b show subject-balanced simulation residuals. Panel c shows the LCModel fit rate for the five UCSF subjects. Seeds are averaged within each subject before mean and sample SD are calculated across subjects. Incomplete LCModel runs are supported: all currently available CRLB maps are used and missing fits produce a warning.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'walinet').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

CONFIG = PROJECT_ROOT / 'configs/PaperFig1/model_comparison.yaml'
U20STAR_RESULTS = PROJECT_ROOT / 'results/PaperFig1/model_comparison_7T_U20star_standard_simulator.h5'
WALINET_STYLE_CONFIG = PROJECT_ROOT / 'configs/PaperFig1/model_comparison_walinet_style.yaml'
WALINET_STYLE_RESULTS = PROJECT_ROOT / 'results/PaperFig1/model_comparison_7T_WALINET_style.h5'
RUN_EVALUATION = False  # Existing HDF5 results are loaded by default.
RUN_WALINET_STYLE_EVALUATION = False
OVERWRITE = False
CRLB_THRESHOLD = 20.0
SIMULATION_LOG_SCALE = True
FIGSIZE = (18.0, 4.8)
SAVE_PDF = PROJECT_ROOT / 'SavedGraphics/7T_quantitative.pdf'  # or None

BASE_DIR = PROJECT_ROOT / 'data/7T/B0corrected_wo_LipidMask'
SUBJECTS = [f'UCSF/Vol{i}' for i in range(1, 6)]
METHOD_GROUPS = {
    'U20*': (
        'LCModelU20_WALINET_Calibration_S1',
        'LCModelU20_WALINET_Calibration_S2',
        'LCModelU20_WALINET_Calibration_S3',
    ),
    'U20': ('LCModelU20_S1', 'LCModelU20_S2', 'LCModelU20_S3'),
    'Y16': ('LCModelY16_S1', 'LCModelY16_S2', 'LCModelY16_S3'),
   # 'WALINET-style': ('LCModelWALINET_Style',),
    'Full WALRUS': ('LCModelAfterWalrus',),
}
SIMULATION_MODEL_GROUPS = {
    'U20*': (
        'UNet_WALINET_calibrated_seed1',
        'UNet_WALINET_calibrated_seed2',
        'UNet_WALINET_calibrated_seed3',
    ),
    'U20': (
        'UNet_parameter_matched_seed1',
        'UNet_parameter_matched_seed2',
        'UNet_parameter_matched_seed3',
    ),
    'Y16': ('YNet_seed1', 'YNet_seed2', 'YNet_seed3'),
    'WALRUS': ('WALRUS',),
    # 'WALINET-style': ('WALINET_style',),
}
METABOLITES = {
    'tNAA': 'NAA_NAAG',
    'tCr': 'Cr_PCr',
    'tCho': 'GPC_PCh',
    'Glx': 'Glu_Gln',
}


In [ ]:
from walinet.paper_figures.paper_fig1 import run_evaluation
from walinet.paper_figures.paper_fig1.config import load_evaluation_config

evaluation_config = load_evaluation_config(CONFIG)
if RUN_EVALUATION:
    base_results_path = run_evaluation(CONFIG, overwrite=OVERWRITE)
else:
    base_results_path = evaluation_config.output_path
if not base_results_path.is_file():
    raise FileNotFoundError(
        f'{base_results_path} does not exist; set RUN_EVALUATION=True.'
    )
if not U20STAR_RESULTS.is_file():
    raise FileNotFoundError(f'Missing calibrated U-Net results: {U20STAR_RESULTS}')
results_path = (base_results_path, U20STAR_RESULTS)
results_path


In [ ]:
from walinet.paper_figures.lcmodel_quality import (
    collect_seeded_fit_rates,
    seeded_summary_table,
)

seed_fit_rates, subject_fit_rates = collect_seeded_fit_rates(
    base_dir=BASE_DIR,
    subjects=SUBJECTS,
    method_groups=METHOD_GROUPS,
    metabolites=METABOLITES,
    crlb_threshold=CRLB_THRESHOLD,
    allow_missing=True,
)
fit_rate_summary = seeded_summary_table(subject_fit_rates)
fit_rate_summary


In [ ]:
from walinet.paper_figures.quantitative_7t import (
    create_combined_quantitative_figure,
)

figure, simulation_subject_results = create_combined_quantitative_figure(
    results_path,
    subject_fit_rates=subject_fit_rates,
    model_groups=SIMULATION_MODEL_GROUPS,
    logarithmic=SIMULATION_LOG_SCALE,
    figsize=FIGSIZE,
)
if SAVE_PDF is not None:
    save_path = Path(SAVE_PDF).expanduser()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(save_path, format='pdf', bbox_inches='tight')
    print(f'Saved: {save_path.resolve()}')

import matplotlib.pyplot as plt
plt.show()


In [ ]:
# Numerical mean +/- sample SD for every point shown in panels a-c.
# Seeds have already been averaged within each subject.
import numpy as np
import pandas as pd
from walinet.paper_figures.paper_fig1.metrics import (
    METABOLITE_RESIDUAL_METRIC,
    NUISANCE_RESIDUAL_METRIC,
)

summary_rows = []
simulation_panels = (
    ('a', 'Nuisance-normalized residual', NUISANCE_RESIDUAL_METRIC),
    ('b', 'Metabolite-normalized residual', METABOLITE_RESIDUAL_METRIC),
)
for panel, description, metric in simulation_panels:
    print(f'Panel {panel}: {description}')
    metric_rows = simulation_subject_results[
        simulation_subject_results['metric'] == metric
    ]
    for method in ('U20*', 'U20', 'Y16', 'WALRUS'):
        values = metric_rows.loc[
            metric_rows['model_group'] == method, 'mean'
        ].to_numpy(dtype=float)
        mean = float(values.mean())
        std = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        print(f'  {method}: {mean:.6g} +/- {std:.6g} (n={len(values)} subjects)')
        summary_rows.append({
            'panel': panel, 'quantity': description, 'method': method,
            'metabolite': '', 'mean': mean, 'std': std,
            'n_subjects': len(values),
        })

print(f'Panel c: LCModel fit rate (CRLB <= {CRLB_THRESHOLD:g}%)')
for method in ('U20*', 'U20', 'Y16', 'Full WALRUS'):
    for metabolite in METABOLITES:
        values = subject_fit_rates.loc[
            (subject_fit_rates['method'] == method)
            & (subject_fit_rates['metabolite'] == metabolite),
            'fit_rate_percent',
        ].to_numpy(dtype=float)
        if len(values) == 0:
            continue
        mean = float(values.mean())
        std = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        print(
            f'  {method}, {metabolite}: {mean:.6g} +/- {std:.6g}% '
            f'(n={len(values)} subjects)'
        )
        summary_rows.append({
            'panel': 'c', 'quantity': f'Fit rate (CRLB <= {CRLB_THRESHOLD:g}%)',
            'method': method, 'metabolite': metabolite,
            'mean': mean, 'std': std, 'n_subjects': len(values),
        })

point_summary = pd.DataFrame(summary_rows)
summary_path = PROJECT_ROOT / 'results/PaperFig1/7T_quantitative_point_summary.csv'
point_summary.to_csv(summary_path, index=False)
print(f'\nSaved point summary: {summary_path.resolve()}')

# Simulated input reference, independent of the evaluated method.
from walinet.paper_figures.quantitative_7t import reference_subject_results

reference_by_subject = reference_subject_results(simulation_subject_results)
reference_mean = reference_by_subject.mean()
reference_std = reference_by_subject.std(ddof=1)

print('Simulated input reference: ||n||_2 / ||m||_2')
print(f'Mean ± SD across {len(reference_by_subject)} subjects: '
      f'{reference_mean:.6g} ± {reference_std:.6g}')
print('\nSubject means:')
for subject, value in reference_by_subject.items():
    print(f'  {subject}: {value:.6g}')
